# Laboratorio — Clasificadores Probabilísticos: NB, LDA y QDA
## Unidad 2 · Lab 2 · Duración: 1 hora · Modalidad: individual o parejas

**versión: 2025-1  |  modificado: 2026-04-06**

---

<div style="background-color:#F8F9FA; border:2px solid #AEB6BF;
     padding:12px 18px; border-radius:8px; margin:10px 0;">
<strong>🎓 Modo de uso:</strong> Este notebook es compartido por dos cursos.<br><br>
<span style="color:#2E86C1; font-weight:bold;">🔵 Pregrado</span>
— Trabaja el contenido general y los bloques azules. Los bloques amarillos
son opcionales y te darán contexto adicional.<br><br>
<span style="color:#B7950B; font-weight:bold;">🟡 Doctorado</span>
— Trabaja el contenido general y <em>ambos</em> bloques. Los bloques azules
te ofrecen la intuición; los amarillos, la formalización.
</div>

---

### 📋 Instrucciones por audiencia

| | Pregrado | Doctorado |
|--|----------|-----------|
| **Obligatorio** | Partes 1, 2, 3 + preguntas azules | Todo lo anterior + TODOs [PhD] + preguntas amarillas |
| **Opcional** | Bonus azul | Bonus amarillo |
| **Entrega** | .ipynb ejecutado | .ipynb ejecutado |

### 🗺️ Estructura del laboratorio

| Parte | Tema | Tiempo | Audiencia |
|-------|------|--------|-----------|
| 1 | Ejercicio sin computador: Bayes a mano | 10 min | Todos |
| 2 | Implementar y comparar GNB, LDA, QDA | 20 min | Todos |
| 3 | Visualización de fronteras e interpretación | 20 min | Todos |
| 4 | Análisis e interpretación | 10 min | Todos |
| Bonus | Efecto del supuesto de independencia | Opcional | — |

In [ ]:
# ── SETUP — NO MODIFICAR ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import __version__ as sklearn_version
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 100

# dataset: wine (sklearn built-in)  |  generado sintéticamente: no
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name='variedad')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

# Proyección LDA para visualización (2 ejes discriminantes)
lda_viz = LinearDiscriminantAnalysis(n_components=2)
lda_viz.fit(X_train, y_train)
X_viz = lda_viz.transform(X.values)    # proyectar todos los datos
X_viz_tr = lda_viz.transform(X_train)  # para entrenar visualización
X_viz_te = lda_viz.transform(X_test)   # para evaluar visualización

def plot_cm(cm, labels, title='', ax=None):
    """Visualiza una matriz de confusión normalizada."""
    if ax is None: _, ax = plt.subplots(figsize=(5, 4))
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_n, annot=cm, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels,
                linewidths=0.5, ax=ax, cbar=False)
    ax.set_xlabel('Predicción', fontsize=10); ax.set_ylabel('Real', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')

print('✅ Setup completo')
print(f'   numpy {np.__version__} · pandas {pd.__version__} · scikit-learn {sklearn_version}')
print(f'   Wine — train: {X_train.shape}, test: {X_test.shape}, clases: {wine.target_names}')

---
## PARTE 1 — Ejercicio sin computador (10 min) ✏️

<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#2E86C1; font-weight:bold; font-size:1em;">🔵 Pregrado</span><br><br>

Un sommelier quiere clasificar vinos en dos variedades (**A** y **B**) usando solo el nivel de alcohol. Las estadísticas estimadas son:

| Variedad | Prior $\pi$ | Media alcohol $\mu$ | Varianza $\sigma^2$ |
|----------|------------|--------------------|--------------|
| A | 0.6 | 13.0 | 0.5 |
| B | 0.4 | 11.5 | 0.8 |

Se analiza un vino con **alcohol = 12.2**.

**a)** Calcula $P(\text{alcohol}=12.2 \mid \text{variedad}=A)$ usando la fórmula gaussiana.  
**b)** Calcula lo mismo para variedad B.  
**c)** Aplicando Bayes, ¿a qué variedad pertenece el vino? ¿Con qué probabilidad?

<br>
<span style="font-size:0.85em; color:#5D6D7E;">→ ¿Quieres la versión formal? Continúa en el bloque 🟡 Doctorado.</span>
</div>

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#B7950B; font-weight:bold; font-size:1em;">🟡 Doctorado</span><br><br>

Con los mismos datos del bloque azul:

**d)** Encuentra analíticamente la **frontera de decisión** $x^*$ donde $P(A|x) = P(B|x)$. Resuélvela como ecuación cuadrática en $x$ (NB con varianzas distintas).

**e)** Si en cambio usas LDA (asumes $\sigma^2_A = \sigma^2_B = 0.65$, el promedio ponderado), ¿dónde quedaría la frontera? ¿Es lineal?

**f)** ¿Cuánto se desplaza la frontera respecto al caso d)? Interpreta ese desplazamiento.

<br>
<span style="font-size:0.85em; color:#7D6608;">→ La intuición detrás de esta derivación está en el bloque 🔵 Pregrado.</span>
</div>

In [ ]:
# ── Verificación Parte 1 — Ejecutar DESPUÉS de responder en el cuaderno ──
import scipy.stats as ss

x_vino = 12.2
clases = {'A': {'pi': 0.6, 'mu': 13.0, 'var': 0.5},
           'B': {'pi': 0.4, 'mu': 11.5, 'var': 0.8}}

print('=== Verificación Parte 1 ===')
log_posts = {}
for nombre, p in clases.items():
    like = ss.norm.pdf(x_vino, loc=p['mu'], scale=np.sqrt(p['var']))
    log_posts[nombre] = np.log(p['pi']) + np.log(like)
    print(f'  P(x={x_vino}|{nombre}) = {like:.4f}  →  log-posterior = {log_posts[nombre]:.4f}')

pred = max(log_posts, key=log_posts.get)
# Normalizar para obtener probabilidades
lp_vals = np.array(list(log_posts.values()))
lp_vals -= lp_vals.max()  # estabilidad numérica
probs = np.exp(lp_vals) / np.exp(lp_vals).sum()
print(f'\n  Clasificación: variedad {pred}')
print(f'  P(A|x) = {probs[0]:.3f}  ·  P(B|x) = {probs[1]:.3f}')

# [PhD] Frontera de decisión NB (cuadrática)
# P(A|x) = P(B|x) → resolver
# 0.5*(x-muA)²/varA + 0.5*log(varA) - log(piA) = 0.5*(x-muB)²/varB + 0.5*log(varB) - log(piB)
from numpy.polynomial import polynomial as P_np
muA, varA, piA = 13.0, 0.5, 0.6
muB, varB, piB = 11.5, 0.8, 0.4
# Coeficientes de ax² + bx + c = 0
a = 0.5*(1/varA - 1/varB)
b = -(muA/varA - muB/varB)
c = 0.5*(muA**2/varA - muB**2/varB) + 0.5*np.log(varA/varB) - np.log(piA/piB)
roots = np.roots([a, b, c])
print(f'\n[PhD] Fronteras de decisión NB (raíces): {np.sort(roots.real).round(3)}')

# LDA frontera (promedio ponderado de varianzas)
n_A, n_B = 60, 40   # proporcional a priors
var_shared = (n_A * varA + n_B * varB) / (n_A + n_B)
x_lda = (muA + muB)/2 - var_shared/(muA - muB) * np.log(piA/piB)
print(f'[PhD] Frontera de decisión LDA: x* = {x_lda:.3f}')

---
## PARTE 2 — Implementar y comparar GNB, LDA, QDA (20 min)

In [ ]:
# TODO 1: Entrena los tres modelos (GaussianNB, LinearDiscriminantAnalysis,
#         QuadraticDiscriminantAnalysis) sobre X_train, y_train.
#         Guarda los modelos en un dict: modelos = {'GNB': ..., 'LDA': ..., 'QDA': ...}
# Pista: no necesitas Pipeline aquí (GNB y LDA/QDA manejan la escala internamente)

modelos = {}  # Reemplaza con tu implementación
# Tu código aquí:


In [ ]:
# TODO 2: Para cada modelo, calcula en el test set:
#         accuracy, f1_macro y genera el classification_report.
#         Imprime una tabla comparativa.

# Tu código aquí:


In [ ]:
# TODO 3: Evalúa los tres modelos con Stratified 10-Fold CV (scoring='f1_macro')
#         sobre el dataset completo (X, y).
#         Imprime: media ± std para cada modelo.

cv_resultados = {}  # Guarda los scores aquí: {'GNB': array, 'LDA': array, 'QDA': array}
# Tu código aquí:


In [ ]:
# TODO 4 [PhD]: Calcula la matriz de covarianza de cada clase en X_train.
#               Visualiza las 3 matrices con seaborn.heatmap en subplots.
#               ¿Son similares entre sí? ¿Esto favorece a LDA o a QDA?

# Tu código aquí:


In [ ]:
# ─── TESTS DE SANIDAD — PARTE 2 — NO MODIFICAR ───
print('=== Tests de Sanidad — Parte 2 ===')
try:
    assert len(modelos) == 3, f'Se esperan 3 modelos, hay {len(modelos)}'
    assert set(modelos.keys()) == {'GNB', 'LDA', 'QDA'}, 'Los modelos deben llamarse GNB, LDA, QDA'
    print(f'✅ Dict de modelos correcto: {list(modelos.keys())}')
except AssertionError as e:
    print(f'❌ {e}')

try:
    for nombre, modelo in modelos.items():
        y_p = modelo.predict(X_test)
        acc = accuracy_score(y_test, y_p)
        assert 0.0 <= acc <= 1.0, f'{nombre}: accuracy fuera de rango'
        assert len(y_p) == len(y_test), f'{nombre}: tamaño de predicciones incorrecto'
    print('✅ Los 3 modelos predicen correctamente sobre X_test')
except AssertionError as e:
    print(f'❌ {e}')

try:
    assert len(cv_resultados) == 3, 'cv_resultados debe tener 3 entradas'
    for nombre, scores in cv_resultados.items():
        assert len(scores) == 10, f'{nombre}: debe tener 10 scores (10-Fold CV)'
        assert all(0 <= s <= 1 for s in scores), f'{nombre}: scores fuera de [0,1]'
    print('✅ CV con 10 folds — scores en rango válido')
except (AssertionError, TypeError) as e:
    print(f'❌ {e}')

---
## PARTE 3 — Visualización de Fronteras (20 min)

In [ ]:
# TODO 5: Visualiza las fronteras de decisión de los 3 modelos
#         usando la proyección LDA 2D (X_viz_tr, X_viz_te ya están calculados en el setup).
#         Pista: entrena cada modelo sobre X_viz_tr, y_train, luego usa
#         DecisionBoundaryDisplay.from_estimator() para cada uno.
#         Organiza los 3 subplots en una fila (1 x 3).

# Tu código aquí:


In [ ]:
# TODO 6: Genera las matrices de confusión de los 3 modelos sobre X_test
#         usando la función plot_cm del setup.
#         Organiza en 1 fila x 3 columnas.

# Tu código aquí:


In [ ]:
# TODO 7 [PhD]: Implementa NB desde cero para el caso 1D del Ejercicio de Parte 1
#               (un feature: 'alcohol') y verifica que coincide con sklearn GaussianNB.
#               Grafica las distribuciones de cada clase y la frontera de decisión.

# Tu código aquí:


In [ ]:
# ─── TESTS DE SANIDAD — PARTE 3 — NO MODIFICAR ───
print('=== Tests de Sanidad — Parte 3 ===')
try:
    # Verificar que los modelos entrenados en X_viz predicen correctamente
    for nombre, modelo in modelos.items():
        # Reutilizamos el modelo del TODO 5 si está disponible
        pass
    print('✅ Visualización completada (verificación visual requerida)')
except Exception as e:
    print(f'❌ {e}')

# Verificar que los modelos del TODO 2 siguen activos
try:
    for nombre, modelo in modelos.items():
        proba = modelo.predict_proba(X_test)
        assert proba.shape == (len(y_test), 3), f'{nombre}: predict_proba shape incorrecto'
        assert np.allclose(proba.sum(axis=1), 1.0, atol=1e-5), f'{nombre}: probabilidades no suman 1'
    print('✅ predict_proba retorna probabilidades válidas (suman 1, shape correcto)')
except AssertionError as e:
    print(f'❌ {e}')

---
## PARTE 4 — Análisis e Interpretación (10 min)

<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#2E86C1; font-weight:bold; font-size:1em;">🔵 Pregrado</span><br><br>

1. Mirando las fronteras de decisión graficadas, ¿qué diferencia visual notable hay entre GNB y LDA? ¿Y entre LDA y QDA?

2. En el dataset Wine, ¿las tres clases tienen formas similares o distintas? ¿Eso favorece a LDA o a QDA?

3. Si tuvieras solo 20 muestras de entrenamiento (en lugar de ~133), ¿cuál de los tres modelos elegirías y por qué?

<br>
<span style="font-size:0.85em; color:#5D6D7E;">→ ¿Quieres profundizar? Continúa en el bloque 🟡 Doctorado.</span>
</div>

<!-- RESPUESTA MODELO:
P1: GNB genera fronteras más «suaves» o curvas que pueden no seguir bien la geometría real; LDA genera frontera recta; QDA puede curvar la frontera en la dirección que mejor separe clases con covarianzas distintas.
P2: Las covarianzas de las 3 clases en Wine son distintas (shape y orientación diferentes) → QDA tiene ventaja teórica. Si LDA aun así supera en CV es porque con 133 muestras y 13 features QDA puede sobreajustar.
P3: Con 20 muestras → NB o LDA, que tienen muy pocos parámetros. QDA necesitaría estimar 3 matrices de 13x13 = ~780 parámetros con solo 20 puntos, lo cual es inviable.
-->

**Respuesta 1:** *(escribe aquí)*

**Respuesta 2:** *(escribe aquí)*

**Respuesta 3:** *(escribe aquí)*

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#B7950B; font-weight:bold; font-size:1em;">🟡 Doctorado</span><br><br>

4. A partir de las matrices de covarianza del TODO 4, ¿son estadísticamente iguales entre clases? Propón una prueba estadística para verificarlo formalmente (no es necesario implementarla, solo describir el procedimiento).

5. El clasificador NB ignora la correlación entre features. En Wine, la correlación entre `flavanoids` y `total_phenols` es ~0.86. ¿En qué dirección esperas que esto afecte al NB respecto a LDA? ¿El resultado de tu CV confirma esa hipótesis?

6. Propón un experimento para cuantificar cuántas muestras de entrenamiento necesita QDA para superar a LDA en Wine. ¿Cómo lo diseñarías?

<br>
<span style="font-size:0.85em; color:#7D6608;">→ La intuición detrás de esta derivación está en el bloque 🔵 Pregrado.</span>
</div>

<!-- RESPUESTA MODELO:
P4: Test de Box's M para igualdad de matrices de covarianza. H0: Sigma_1 = Sigma_2 = Sigma_3. Si se rechaza, LDA viola su supuesto y QDA es más apropiado. En la práctica Box's M es muy sensible a no-normalidad.
P5: Alta correlación → NB subestima la log-verosimilitud porque asume términos independientes que en realidad son redundantes. Esto puede inflar la confianza del modelo. LDA captura esa correlación en Sigma. Si en CV LDA > NB, confirma la hipótesis.
P6: Curva de aprendizaje comparativa (n_train vs f1 para LDA y QDA). El cruce de las curvas indica el n* donde QDA empieza a ganar. Para Wine con 13 features, se esperaría que n* sea bastante alto (>100 por clase aproximadamente).
-->

**Respuesta 4 [PhD]:** *(escribe aquí)*

**Respuesta 5 [PhD]:** *(escribe aquí)*

**Respuesta 6 [PhD]:** *(escribe aquí)*

---
## BONUS

<div style="background-color:#EBF5FB; border-left:5px solid #2E86C1;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#2E86C1; font-weight:bold; font-size:1em;">🔵 Bonus Pregrado</span><br><br>

Genera un **dataset sintético 2D** donde NB claramente falle y LDA triunfe. Pista: crea dos clases gaussianas con alta correlación entre sus features (usa `np.random.multivariate_normal` con una covarianza no diagonal). Visualiza las fronteras de ambos modelos.

<br>
<span style="font-size:0.85em; color:#5D6D7E;">→ ¿Quieres ir más lejos? Continúa en el bloque 🟡 Doctorado.</span>
</div>

<div style="background-color:#FEFDE7; border-left:5px solid #D4AC0D;
     padding:14px 18px; border-radius:6px; margin:12px 0;">
<span style="color:#B7950B; font-weight:bold; font-size:1em;">🟡 Bonus Doctorado</span><br><br>

Implementa **LDA desde cero** (sin sklearn) para clasificación binaria en 2D:
1. Estima $\mu_0$, $\mu_1$ y $\Sigma_W$ (within-class covariance) de los datos.
2. Calcula $w = \Sigma_W^{-1}(\mu_1 - \mu_0)$ y el umbral $c = w^T(\mu_0 + \mu_1)/2$.
3. Predice: $\hat{y} = 1$ si $w^T x > c$, else 0.
4. Verifica que coincide con `sklearn.LinearDiscriminantAnalysis`.

<br>
<span style="font-size:0.85em; color:#7D6608;">→ La intuición detrás de esta derivación está en el bloque 🔵 Pregrado.</span>
</div>

In [ ]:
# Espacio de trabajo para el Bonus


---
## ✅ Checklist de entrega

### Pregrado
- [ ] Parte 1: respondí en el cuaderno antes de ejecutar la verificación
- [ ] TODOs 1–3 completados y celdas ejecutadas
- [ ] TODOs 5–6 con visualizaciones generadas
- [ ] Tests de sanidad: todos ✅ PASS
- [ ] Parte 4: preguntas 1–3 respondidas

### Doctorado (adicional)
- [ ] TODO 4: matrices de covarianza calculadas y visualizadas
- [ ] TODO 7: NB desde cero verificado contra sklearn
- [ ] Parte 4: preguntas 4–6 respondidas
- [ ] Bonus completado

### Ambas audiencias
- [ ] El notebook ejecuta de arriba a abajo sin errores
- [ ] Entrego el archivo: `ML_U2_Lab02_Apellido_Nombre.ipynb`